# 05 - Objects in Motion

## Imports

In [1]:
import numpy as np

## How Motion Changes Detection 

In our previous notebook, we were dealing with what is called **static detection** - we assume that the images we take never changed. 

While at times this assumption could hold true, most of the time, our robot is going to be on the move, and therefore our Limelights are not going to be seeing the same frame. They could be seeing the same object, but that position will change. 

How do we track this change? 

## Pixel Motion

### Displacement

Let's get a little more granular than talking about objects - let's talk about the building blocks of images, pixels. 

If the frame never changes, then a given pixel will always correspond to a given object.

However, if the frame changes, a given pixel may not correspond to the same object. To fix this, we will start tracking pixel movement, not object movement. 

This adds a *temporal* component to object detection: we now cover both space and time. 

Let's show an example of how this allows for better object detection with motion: 

![](./ref_imgs/basic_frame_displacement.png)

As we can see above, we pick a single pixel in our original frame at time $t$, noting its $(x, y)$ position. Then, when movement occurs, we get a new frame at time $(t + dt)$, and we have a new pixel position $(x + dx, y + dy)$. 

Note that $d$ denotes a change, or **displacement**, and can also be denoted with $\Delta$.

That displacement *is* motion. 

How do we use this to track objects?

### Tracking Pixels Over Time: Optical Flow

The core logic of how we translate this to detecting objects is fortunately intuitive: we find the pixels corresponding to the object we want, and track those pixels over time. 

This only works under a key assumption of **brightness constancy**, written as 

$$ I(x, y, t) = I(x + dx, y + dy, t + dt) $$

where $I$ is the intensity of the pixel. Essentially, this tells us that intensity never changes, meaning that if a pixel at a given spot changes intensity, it is *only* because there is a different pixel there. 

Otherwise, if this assumption fails, we cannot reliably use pixel tracking as a form of object detection. Any pixel changes could be a result of object color change, brightness change due to different exposure of the camera, etc. 

For pixel tracking, or in this case, **optical flow**, there are two different approaches to how we can select the pixels to track. Note that neither of these approaches tell us what the object is, it just tells us where the pixels go. We have to determine what the objects are later on.


#### Sparse Optical Flow

In **sparse optical flow**, we select only a few choice pixels on the objects of interest - think important features like edges. 

Let's look at an example: 

![](./ref_imgs/sparse_optical_flow.png)

As you can see, for the people in the image, we have only selected a subset of pixels on each person. The arrows show the direction of the displacement (e.g., an arrow pointing left means the originally tracked pixel has moved left), and therefore allow us to track the people! 

Algorithms like the Lucas-Kanade method leverage sparse optical flow, which is nice because it requires much less computation, but also has its flaws in that we must be precise in our original pixel selection to ensure we are tracking our objects properly. 

#### Dense Optical Flow

In **dense optical flow**, we track the motion of *all* the pixels. 

Let's see this in action with an image below: 

![](./ref_imgs/dense_optical_flow.png)

That is a lot of arrows! Algorithms like the Farneback method leverage dense optical flow, which is nice because we don't have to be good at selecting pixels since we use all of them, but this comes at a greater computational cost. 

## From Pixel Motion to Object Tracking

We covered optical flow above, but noted that we don't actually track our objects of interest. Sure, our eyes can see that the pixels are tracking the proper objects, but our robot and our camera have no idea. 

We can leverage our principles from our static object detection to identify our objects of interest. 

### Object Detection in Motion

#### Bounding Boxes and Class Labels

We can go back to our object detection basics of having bounding boxes for localization and labels for classification! From the start, we draw our bounding boxes around our objects of interest, and we assign the class labels.

How do we track the bounding boxes and labels over time? 

#### Matching Frames

We can match the frames based on several different factors, like the position, motion, size, or appearance. 

For instance, we can match using distance as a starting algorithm: the closest box in the new frame to the original box in the old frame is the moved box. 

A slightly better approach would be using IoU, where the highest overlap between two boxes constitutes a match. 

Ultimately, these approaches can work nicely, but what happens if an object no longer appears in the frame? 

#### Occlusion